# ROGII Wellbore Geology Prediction Stronger Submission

Experimental v2 notebook. Run this notebook on Kaggle with the competition data attached. It reads the competition files from `/kaggle/input/`, trains only from notebook-visible tabular data, validates submission schema, and writes `/kaggle/working/submission.csv`.

Public leaderboard feedback should be treated as a sanity check only; model choice should prioritize well-level and hidden-interval validation.


In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

SEED = 2026
TARGET = "TVT"
SUBMISSION_TARGET = "tvt"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_INPUT_DIR = KAGGLE_INPUT_ROOT / "rogii-wellbore-geology-prediction"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")

# Keep validation available for local/Kaggle diagnostics, but avoid making it a runtime blocker.
RUN_LOCAL_VALIDATION = False
VALIDATION_FOLDS = 3
VALIDATION_ESTIMATORS = 260
ENSEMBLE_SEEDS = [2026, 2027, 2028]
ENABLE_XGBOOST_ENSEMBLE = False
ENABLE_CATBOOST_ENSEMBLE = False
CLIP_PREDICTIONS = True
CLIP_QUANTILES = (0.001, 0.999)
SMOOTH_HIDDEN_INTERVALS = True
SMOOTH_WINDOW = 5
SMOOTH_BLEND = 0.15

HORIZONTAL_COMMON_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
HORIZONTAL_TRAIN_ONLY_COLS = {"ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"}
TYPEWELL_COMMON_NUMERIC_COLS = ["TVT", "GR"]


def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0].split(".", 1)[0]


def find_input_dir() -> Path:
    candidates: list[Path] = []
    if (PREFERRED_INPUT_DIR / "sample_submission.csv").exists():
        candidates.append(PREFERRED_INPUT_DIR)
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(path.parent for path in sorted(KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")))

    unique_candidates = list(dict.fromkeys(candidates))
    valid_candidates = [path for path in unique_candidates if (path / "train").is_dir() and (path / "test").is_dir()]
    if len(valid_candidates) == 1:
        return valid_candidates[0]
    if len(valid_candidates) > 1:
        names = [str(path) for path in valid_candidates]
        raise ValueError(f"Multiple possible competition input directories found: {names}")

    available = sorted(str(path) for path in KAGGLE_INPUT_ROOT.glob("*")) if KAGGLE_INPUT_ROOT.exists() else []
    nested_samples = sorted(str(path) for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")) if KAGGLE_INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "Could not find a Kaggle input directory containing sample_submission.csv, train/, and test/. "
        f"Available /kaggle/input entries: {available}. "
        f"Nested sample_submission.csv files found: {nested_samples}"
    )


INPUT_DIR = find_input_dir()
print(f"Using competition input directory: {INPUT_DIR}")


def list_files(split: str, kind: str) -> list[Path]:
    folder = INPUT_DIR / split
    patterns = {"horizontal": "*__horizontal_well.csv", "typewell": "*__typewell.csv"}
    if kind not in patterns:
        raise ValueError(f"Unknown file kind: {kind}")
    files = sorted(folder.glob(patterns[kind]))
    if not files:
        raise FileNotFoundError(f"No {kind} files found under {folder}")
    return files


def load_horizontal(split: str) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in list_files(split, "horizontal"):
        raw = pd.read_csv(path)
        keep = [col for col in HORIZONTAL_COMMON_COLS if col in raw.columns]
        if TARGET in raw.columns:
            keep.append(TARGET)
        df = raw[keep].copy()
        df.insert(0, "well_id", well_id_from_path(path))
        df.insert(1, "row_id", np.arange(len(df), dtype=np.int32))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)


def load_typewell_summary(split: str) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    for path in list_files(split, "typewell"):
        raw = pd.read_csv(path)
        row: dict[str, float | int | str] = {"well_id": well_id_from_path(path), "typewell_rows": int(len(raw))}
        for col in TYPEWELL_COMMON_NUMERIC_COLS:
            if col not in raw.columns:
                continue
            s = pd.to_numeric(raw[col], errors="coerce")
            prefix = f"typewell_{col}"
            row[f"{prefix}_mean"] = float(s.mean())
            row[f"{prefix}_std"] = float(s.std())
            row[f"{prefix}_min"] = float(s.min())
            row[f"{prefix}_max"] = float(s.max())
            row[f"{prefix}_range"] = float(s.max() - s.min())
            row[f"{prefix}_q10"] = float(s.quantile(0.10))
            row[f"{prefix}_q25"] = float(s.quantile(0.25))
            row[f"{prefix}_q50"] = float(s.quantile(0.50))
            row[f"{prefix}_q75"] = float(s.quantile(0.75))
            row[f"{prefix}_q90"] = float(s.quantile(0.90))
        rows.append(row)
    return pd.DataFrame(rows)


def build_prediction_ids(df: pd.DataFrame) -> pd.Series:
    return df["well_id"].astype(str) + "_" + df["row_id"].astype(str)


def add_typewell_features(work: pd.DataFrame, split: str) -> pd.DataFrame:
    type_summary = load_typewell_summary(split)
    if not type_summary.empty:
        work = work.merge(type_summary, on="well_id", how="left")
    if "GR" in work.columns and "typewell_GR_mean" in work.columns:
        work["GR_minus_typewell_GR_mean"] = work["GR"] - work["typewell_GR_mean"]
        work["GR_minus_typewell_GR_q50"] = work["GR"] - work.get("typewell_GR_q50", work["typewell_GR_mean"])
        denom = work["typewell_GR_std"].replace(0, np.nan)
        work["GR_typewell_z"] = (work["GR"] - work["typewell_GR_mean"]) / denom
    return work


def add_position_and_context_features(work: pd.DataFrame) -> pd.DataFrame:
    group = work.groupby("well_id", sort=False)
    rows_in_well = group["row_id"].transform("max").astype(float) + 1.0
    denom = (rows_in_well - 1.0).replace(0, 1.0)
    work["rows_in_well"] = rows_in_well
    work["row_frac"] = work["row_id"] / denom
    work["rows_from_end"] = rows_in_well - work["row_id"] - 1.0

    if "TVT_input" in work.columns:
        known = work["TVT_input"].notna()
        work["tvt_input_missing"] = (~known).astype(np.int8)
        work["tvt_input_known"] = known.astype(np.int8)
        known_row = work["row_id"].where(known)
        first_known = known_row.groupby(work["well_id"], sort=False).transform("min")
        last_known = known_row.groupby(work["well_id"], sort=False).ffill()
        next_known = known_row.groupby(work["well_id"], sort=False).bfill()
        work["rows_since_last_tvt_input"] = (work["row_id"] - last_known).fillna(0.0)
        work["rows_to_next_tvt_input"] = (next_known - work["row_id"]).fillna(0.0)
        work["known_row_frac"] = (last_known.fillna(first_known).fillna(0.0) / denom).clip(0.0, 1.0)
        hidden_denom = (rows_in_well - last_known).replace(0, np.nan)
        work["hidden_context_position"] = ((work["row_id"] - last_known) / hidden_denom).fillna(0.0).clip(0.0, 1.0)
        if "MD" in work.columns:
            known_md = work["MD"].where(known)
            last_known_md = known_md.groupby(work["well_id"], sort=False).ffill()
            work["md_since_last_tvt_input"] = (work["MD"] - last_known_md).fillna(0.0)
        else:
            work["md_since_last_tvt_input"] = 0.0
    else:
        work["tvt_input_missing"] = 0
        work["tvt_input_known"] = 0
        work["rows_since_last_tvt_input"] = 0.0
        work["rows_to_next_tvt_input"] = 0.0
        work["known_row_frac"] = 0.0
        work["hidden_context_position"] = 0.0
        work["md_since_last_tvt_input"] = 0.0
    return work


def add_sequence_features(work: pd.DataFrame) -> pd.DataFrame:
    base_numeric = [col for col in ["MD", "X", "Y", "Z", "GR"] if col in work.columns]
    for col in base_numeric:
        work[f"{col}_missing"] = work[col].isna().astype(np.int8)
        work[col] = pd.to_numeric(work[col], errors="coerce")
        work[col] = work.groupby("well_id", sort=False)[col].transform(lambda s: s.ffill().bfill())
        work[col] = work[col].fillna(work[col].median())
        group_col = work.groupby("well_id", sort=False)[col]
        first = group_col.transform("first")
        last = group_col.transform("last")
        work[f"{col}_from_start"] = work[col] - first
        work[f"{col}_from_end"] = last - work[col]
        work[f"{col}_range_in_well"] = group_col.transform("max") - group_col.transform("min")
        work[f"{col}_diff1"] = group_col.diff().fillna(0.0)
        work[f"{col}_diff3"] = (work[col] - group_col.shift(3)).fillna(0.0)

    md_diff = work.get("MD_diff1", pd.Series(0.0, index=work.index)).replace(0, np.nan)
    for col in ["X", "Y", "Z", "GR"]:
        if f"{col}_diff1" in work.columns:
            work[f"{col}_slope_per_md"] = (work[f"{col}_diff1"] / md_diff).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if all(col in work.columns for col in ["X_diff1", "Y_diff1", "Z_diff1"]):
        work["step_distance_3d"] = np.sqrt(work["X_diff1"] ** 2 + work["Y_diff1"] ** 2 + work["Z_diff1"] ** 2)
        work["step_distance_xy"] = np.sqrt(work["X_diff1"] ** 2 + work["Y_diff1"] ** 2)
        work["vertical_delta_abs"] = work["Z_diff1"].abs()
        work["step_distance_3d_per_md"] = (work["step_distance_3d"] / md_diff).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    for col in base_numeric:
        group_col = work.groupby("well_id", sort=False)[col]
        for window in [3, 7, 15, 31]:
            rolled = group_col.rolling(window, min_periods=1)
            work[f"{col}_roll{window}_mean"] = rolled.mean().reset_index(level=0, drop=True)
            work[f"{col}_roll{window}_std"] = rolled.std().reset_index(level=0, drop=True).fillna(0.0)
            work[f"{col}_roll{window}_min"] = rolled.min().reset_index(level=0, drop=True)
            work[f"{col}_roll{window}_max"] = rolled.max().reset_index(level=0, drop=True)

    agg_cols = [col for col in ["MD", "X", "Y", "Z", "GR", "step_distance_3d", "step_distance_xy"] if col in work.columns]
    group = work.groupby("well_id", sort=False)
    for col in agg_cols:
        g = group[col]
        work[f"well_{col}_mean"] = g.transform("mean")
        work[f"well_{col}_std"] = g.transform("std").fillna(0.0)
        work[f"well_{col}_min"] = g.transform("min")
        work[f"well_{col}_max"] = g.transform("max")
        work[f"well_{col}_range"] = work[f"well_{col}_max"] - work[f"well_{col}_min"]
    if "step_distance_3d" in work.columns:
        work["well_path_length_3d"] = group["step_distance_3d"].transform("sum")
    return work


def add_features(df: pd.DataFrame, split: str) -> pd.DataFrame:
    keep = ["well_id", "row_id"] + [col for col in HORIZONTAL_COMMON_COLS if col in df.columns]
    if TARGET in df.columns:
        keep.append(TARGET)
    work = df[keep].copy().sort_values(["well_id", "row_id"]).reset_index(drop=True)
    work = add_position_and_context_features(work)
    work = add_sequence_features(work)
    work = add_typewell_features(work, split)
    return work


def select_common_numeric_features(train: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    excluded = {TARGET, "TVT_input", "well_id", "id"}
    common = [col for col in train.columns if col in test.columns]
    features = [
        col for col in common
        if col not in excluded
        and pd.api.types.is_numeric_dtype(train[col])
        and pd.api.types.is_numeric_dtype(test[col])
    ]
    if not features:
        raise ValueError("No common numeric features are available for training and prediction.")
    return features


def clean_feature_matrix(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_x = train[features].replace([np.inf, -np.inf], np.nan)
    test_x = test[features].replace([np.inf, -np.inf], np.nan)
    medians = train_x.median(numeric_only=True).fillna(0.0)
    train_x = train_x.fillna(medians).fillna(0.0)
    test_x = test_x.fillna(medians).fillna(0.0)
    return train_x, test_x


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)) ** 2)))


def make_lightgbm(seed: int, n_estimators: int):
    from lightgbm import LGBMRegressor
    return LGBMRegressor(
        objective="regression",
        n_estimators=n_estimators,
        learning_rate=0.035,
        num_leaves=96,
        min_child_samples=80,
        subsample=0.88,
        colsample_bytree=0.82,
        reg_alpha=0.05,
        reg_lambda=0.15,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )


def make_fallback(seed: int):
    from sklearn.ensemble import HistGradientBoostingRegressor
    return HistGradientBoostingRegressor(
        max_iter=420,
        learning_rate=0.04,
        l2_regularization=0.02,
        random_state=seed,
    )


def predict_seed_ensemble(train_x: pd.DataFrame, y: np.ndarray, test_x: pd.DataFrame) -> tuple[np.ndarray, list[str]]:
    predictions: list[np.ndarray] = []
    model_names: list[str] = []
    try:
        for seed in ENSEMBLE_SEEDS:
            model = make_lightgbm(seed, n_estimators=760)
            model.fit(train_x, y)
            predictions.append(np.asarray(model.predict(test_x), dtype=float))
            model_names.append(f"lightgbm_seed_{seed}")
    except Exception as lightgbm_error:
        print(f"LightGBM ensemble unavailable; falling back to HistGradientBoosting. Error: {lightgbm_error}")
        model = make_fallback(SEED)
        model.fit(train_x, y)
        predictions.append(np.asarray(model.predict(test_x), dtype=float))
        model_names.append("hist_gradient_boosting")

    if ENABLE_XGBOOST_ENSEMBLE:
        try:
            from xgboost import XGBRegressor
            xgb = XGBRegressor(
                n_estimators=420,
                learning_rate=0.035,
                max_depth=7,
                subsample=0.88,
                colsample_bytree=0.82,
                objective="reg:squarederror",
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
            )
            xgb.fit(train_x, y)
            predictions.append(np.asarray(xgb.predict(test_x), dtype=float))
            model_names.append("xgboost_optional")
        except Exception as error:
            print(f"Optional XGBoost model skipped: {error}")
    else:
        print("Optional XGBoost ensemble disabled for runtime control.")

    if ENABLE_CATBOOST_ENSEMBLE:
        try:
            from catboost import CatBoostRegressor
            cat = CatBoostRegressor(
                iterations=420,
                learning_rate=0.035,
                depth=8,
                loss_function="RMSE",
                random_seed=SEED,
                verbose=False,
                allow_writing_files=False,
            )
            cat.fit(train_x, y)
            predictions.append(np.asarray(cat.predict(test_x), dtype=float))
            model_names.append("catboost_optional")
        except Exception as error:
            print(f"Optional CatBoost model skipped: {error}")
    else:
        print("Optional CatBoost ensemble disabled for runtime control.")

    pred = np.mean(np.vstack(predictions), axis=0)
    return pred, model_names


def clip_predictions(pred: np.ndarray, y: np.ndarray) -> np.ndarray:
    if not CLIP_PREDICTIONS:
        print("Prediction clipping disabled.")
        return pred
    low, high = np.quantile(y, CLIP_QUANTILES)
    print(f"Prediction clipping enabled: quantiles={CLIP_QUANTILES}, bounds=({low:.4f}, {high:.4f})")
    return np.clip(pred, low, high)


def smooth_submission_predictions(submission: pd.DataFrame) -> pd.DataFrame:
    if not SMOOTH_HIDDEN_INTERVALS:
        print("Hidden-interval smoothing disabled.")
        return submission
    work = submission.copy()
    parsed = work["id"].astype(str).str.rsplit("_", n=1, expand=True)
    work["_well_id"] = parsed[0]
    work["_row_id"] = pd.to_numeric(parsed[1], errors="coerce")
    work["_original_order"] = np.arange(len(work))
    work = work.sort_values(["_well_id", "_row_id", "_original_order"])
    smoothed = (
        work.groupby("_well_id", sort=False)[SUBMISSION_TARGET]
        .transform(lambda s: s.rolling(SMOOTH_WINDOW, min_periods=1, center=True).mean())
    )
    work[SUBMISSION_TARGET] = (1.0 - SMOOTH_BLEND) * work[SUBMISSION_TARGET] + SMOOTH_BLEND * smoothed
    work = work.sort_values("_original_order").drop(columns=["_well_id", "_row_id", "_original_order"])
    print(f"Hidden-interval smoothing enabled: window={SMOOTH_WINDOW}, blend={SMOOTH_BLEND}")
    return work


def validate_submission(output_path: Path, sample: pd.DataFrame) -> None:
    if not output_path.exists():
        raise FileNotFoundError(f"Submission file was not written: {output_path}")
    sub = pd.read_csv(output_path)
    problems: list[str] = []
    if list(sub.columns) != list(sample.columns):
        problems.append(f"columns {list(sub.columns)} do not match {list(sample.columns)}")
    if SUBMISSION_TARGET not in sub.columns:
        problems.append(f"prediction column must be named {SUBMISSION_TARGET!r}")
    if len(sub) != len(sample):
        problems.append(f"row count {len(sub)} does not match {len(sample)}")
    if "id" in sub.columns and not sub["id"].equals(sample["id"]):
        problems.append("id order does not match sample_submission.csv")
    if SUBMISSION_TARGET in sub.columns:
        values = sub[SUBMISSION_TARGET].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            problems.append("submission contains NaN or infinite predictions")
    if problems:
        raise ValueError(f"Submission validation failed: {problems}")


def run_group_validation(train: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    from sklearn.model_selection import GroupKFold

    work = train.loc[train[TARGET].notna()].copy()
    x_all = work[features].replace([np.inf, -np.inf], np.nan)
    y_all = work[TARGET].to_numpy(dtype=float)
    groups = work["well_id"].to_numpy()
    medians = x_all.median(numeric_only=True).fillna(0.0)
    x_all = x_all.fillna(medians).fillna(0.0)

    rows: list[dict[str, float | int | str]] = []
    splitter = GroupKFold(n_splits=VALIDATION_FOLDS)
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(x_all, y_all, groups), start=1):
        model = make_lightgbm(SEED + fold, n_estimators=VALIDATION_ESTIMATORS)
        model.fit(x_all.iloc[train_idx], y_all[train_idx])
        pred = np.asarray(model.predict(x_all.iloc[valid_idx]), dtype=float)
        hidden_mask = work.iloc[valid_idx]["tvt_input_missing"].to_numpy(dtype=bool)
        rows.append({
            "fold": fold,
            "valid_wells": int(pd.Series(groups[valid_idx]).nunique()),
            "valid_rows": int(len(valid_idx)),
            "hidden_rows": int(hidden_mask.sum()),
            "group_rmse": rmse(y_all[valid_idx], pred),
            "hidden_interval_rmse": rmse(y_all[valid_idx][hidden_mask], pred[hidden_mask]) if hidden_mask.any() else np.nan,
        })
        print(rows[-1])
    return pd.DataFrame(rows)


sample_path = INPUT_DIR / "sample_submission.csv"
if not sample_path.exists():
    raise FileNotFoundError(f"Missing sample submission: {sample_path}")

sample = pd.read_csv(sample_path)
if list(sample.columns) != ["id", SUBMISSION_TARGET]:
    raise ValueError(f"Expected sample submission columns ['id', '{SUBMISSION_TARGET}'], got {list(sample.columns)}")

raw_train = load_horizontal("train")
raw_test = load_horizontal("test")
train = add_features(raw_train, "train")
test = add_features(raw_test, "test")

if TARGET not in train.columns:
    raise ValueError(f"Training data does not contain target column {TARGET!r}")
train = train.loc[train[TARGET].notna()].copy()
if train.empty:
    raise ValueError("No non-missing TVT targets are available for training.")

test["id"] = build_prediction_ids(test)
features = select_common_numeric_features(train, test)
train_x, test_x = clean_feature_matrix(train, test, features)
y = train[TARGET].to_numpy(dtype=float)

print("Feature policy: using train/test-compatible horizontal columns MD/X/Y/Z/GR/TVT_input context only.")
print("Feature policy: train-only horizontal columns and typewell Geology are excluded.")
print(f"Train rows: {len(train):,}; test rows: {len(test):,}; features: {len(features)}")
print(f"Train hidden-context rows: {int(train['tvt_input_missing'].sum()):,}")
print(f"Test hidden-context rows: {int(test['tvt_input_missing'].sum()):,}")

if RUN_LOCAL_VALIDATION:
    validation = run_group_validation(train, features)
    print("Validation summary:")
    print(validation.to_string(index=False))
else:
    print("Local validation skipped in this submission run. Set RUN_LOCAL_VALIDATION=True for well-level and hidden-interval diagnostics.")

predictions, model_names = predict_seed_ensemble(train_x, y, test_x)
if not np.isfinite(predictions).all():
    raise ValueError("Model produced non-finite test predictions.")
predictions = clip_predictions(predictions, y)

pred = pd.DataFrame({"id": test["id"].to_numpy(), SUBMISSION_TARGET: predictions})
submission = sample[["id"]].merge(pred, on="id", how="left", validate="one_to_one")
if submission[SUBMISSION_TARGET].isna().any():
    missing_ids = submission.loc[submission[SUBMISSION_TARGET].isna(), "id"].head(10).tolist()
    raise ValueError(f"Could not generate predictions for all sample IDs. First missing IDs: {missing_ids}")

submission = smooth_submission_predictions(submission)
submission = submission[list(sample.columns)]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
validate_submission(OUTPUT_PATH, sample)

print("Kaggle Notebook stronger submission completed.")
print(f"Models: {model_names}")
print(f"Submission rows: {len(submission)}")
print(f"Prediction range: {submission[SUBMISSION_TARGET].min():.4f} to {submission[SUBMISSION_TARGET].max():.4f}")
print(f"Wrote: {OUTPUT_PATH}")
print("Validation passed: columns, row count, id order, and finite tvt predictions.")
